# Yambda AutoFE — локальный запуск (слабый ноутбук, CPU)

Подготовка (один раз, в терминале в папке репозитория):
```bash
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt jupyter
jupyter notebook notebooks/local_run.ipynb
```
Нужно: Python ≥ 3.10, интернет (скачивается Yambda-50M с HuggingFace, ~1–2 ГБ на диске).
Память: для 1000 пользователей хватает ~4 ГБ свободной RAM; 2000 пользователей — ~6–8 ГБ.

Порядок: тесты → smoke на 200 пользователях (проверка, что всё обучается) → основной прогон → отчёт.

In [ ]:
N_USERS = 1000          # увеличьте до 2000-3000, если хватает памяти
SMOKE_USERS = 200

import os, sys, subprocess, time
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('repo:', os.getcwd(), '| python:', sys.executable, '| cpu:', os.cpu_count())
try:
    mem = [l for l in open('/proc/meminfo') if l.startswith(('MemTotal', 'MemAvailable'))]
    print(''.join(mem))
except OSError:
    pass
import importlib
for m in ['numpy', 'pandas', 'pyarrow', 'sklearn', 'lightgbm', 'huggingface_hub', 'matplotlib', 'tabulate']:
    try:
        print(m, importlib.import_module(m).__version__)
    except Exception as e:
        print(m, 'НЕ УСТАНОВЛЕН -> pip install -r requirements.txt')

def run(*args):
    """Запуск скрипта тем же python, что и у ноутбука, с потоковым выводом."""
    t = time.time()
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'Warning' not in line:
            print(line, end='')
    p.wait()
    print(f'--- exit code {p.returncode}, {time.time() - t:.0f} s')
    assert p.returncode == 0, 'ошибка, пришлите вывод выше'

In [ ]:
# 1) тесты на утечки (~15 с)
run('-m', 'pytest', '-q', 'tests')

In [ ]:
# 2) SMOKE: весь пайплайн на 200 реальных пользователях (первый раз скачается датасет)
run('scripts/run_experiments.py', '--smoke', '--n-users', str(SMOKE_USERS), '--out', f'results_smoke/u{SMOKE_USERS}')
run('scripts/make_report.py', 'results_smoke')

Если smoke дошёл до `done` и AUC заметно > 0.5 — запускаем основной прогон.

In [ ]:
# 3) основной прогон (LightGBM на CPU)
run('scripts/run_experiments.py', '--n-users', str(N_USERS), '--out', f'results/u{N_USERS}')

In [ ]:
# 4) таблицы и графики -> results/report/
run('scripts/make_report.py', 'results')
print(open('results/report/tables.md').read()[:4000])